# 2.5.1. 一个简单的例子¶


In [25]:
import torch
x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

In [26]:
# 开启梯度跟踪，此时pytorch会记录你后面所有运算过程，建立计算图（Computational Graph）
x.requires_grad_(True)  # ← 开启梯度跟踪
x.grad                  # None (还没求导)
y = 2 * torch.dot(x, x) # xTx =  # y = 2(x·x) = 2(x₁² + x₂² + x₃² + x₄²)
y                       

tensor(28., grad_fn=<MulBackward0>)

In [27]:
y.backward() # ← 根据记录，自动求导 求 ∂y​ / ∂xi  = 2⋅2xi = 4xi; backward() = 计算导数（梯度），存到 x.grad
x.grad, x , x.grad == 4 * x   # ← 获取梯度，查看结果 [0, 4, 8, 12]; ✅ 新增，存储导数 [0, 4, 8, 12]
             

(tensor([ 0.,  4.,  8., 12.]),
 tensor([0., 1., 2., 3.], requires_grad=True),
 tensor([True, True, True, True]))

In [ ]:
x.grad.zero_()         # 默认情况下，pytorch会把所有的梯度给你累积起来，这里清空之前x.grad里面储存的梯度 变成 x.grad = [0,0,0,0]
y = x.sum()            # y = x1+x2+x3+x4  y = 0+1+2+3 = 6
y.backward()           # 梯度求导 ∂y​ / ∂xi 又因为 y = x1+x2+x3+x4 所以 ∇y=[1,1,1,1]
x.grad


tensor([1., 1., 1., 1.])

# 2.5.2. 非标量变量的反向传播

In [ ]:
# 对非标量调用backward需要传入一个gradient参数，该参数指定微分函数关于self的梯度。
# 本例只想求偏导数的和，所以传递一个1的梯度是合适的
x.grad.zero_()
y = x * x
# 等价于y.backward(torch.ones(len(x)))
y.sum().backward()
x.grad
# 照理说，如果y和x都是向量，求导出来的结果应该是矩阵，可是在深度学习里面很少用到矩阵，所以用sum()把y变成标量

tensor([0., 2., 4., 6.])

# 2.5.3. 分离计算


In [ ]:
# detach() = 把张量从计算图里剪出来，当成普通常数用，求导时就不会追溯到它之前的计算了。
# 正常情况下，pytorch会记住计算过程：x → x*x → y → y*x → z 求导的时候也会根据链式法则追溯回去
# u = y.detach() 把 y 的数值复制给 u，但切断和之前计算的联系
# x → x*x → y； u * x → z （u（只是个普通数字，和x没关系了）
x.grad.zero_()
y = x * x
u = y.detach()
z = u * x

z.sum().backward()
x.grad == u

tensor([True, True, True, True])

In [ ]:
x.grad.zero_()
y.sum().backward() # y = x² 求导就是 2x
x.grad == 2 * x

tensor([True, True, True, True])

# 2.5.4. Python控制流的梯度计算

In [ ]:
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c
a = torch.randn(size=(), requires_grad=True) # size为空就是一个标量
d = f(a)
d.backward()
a.grad == d / a

tensor(True)